In [1]:
import numpy as np 
import matplotlib.pyplot as plt
import nidaqmx
import matplotlib.animation as animation
from nidaqmx.constants import AcquisitionType
import datetime as datetime
from IPython.display import clear_output
import time

In [ ]:
#para saber el ID de la placa conectada (DevX)
system = nidaqmx.system.System.local()
for device in system.devices:
    print(device)

In [ ]:
def save_measurement(t, V, comment=""):
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"measurement_{timestamp}.csv"
    
    header = f"Comment: {comment}\nTime (s),Voltage (V)"
    data = np.column_stack((t, V))
    
    np.savetxt(filename, data, delimiter=",", header=header, comments="")
    print(f"Saved to {filename}")

In [ ]:
sample_rate = 25000
samples_per_read = 2500
buffer_size = 25000        # se muestra solamente un segundo de data
duration_seconds = 5
channel_name = "Dev1/ai0"

task = nidaqmx.Task()
task.ai_channels.add_ai_voltage_chan(channel_name)
task.timing.cfg_samp_clk_timing(
    sample_rate,
    sample_mode=AcquisitionType.CONTINUOUS,
    samps_per_chan=samples_per_read
)
task.start()

display_buffer = np.zeros(buffer_size)
all_data = []

# negativo pq es el segundo que ya paso jaj
time_axis = np.linspace(-buffer_size/sample_rate, 0, buffer_size)

start_time = time.time()
try:
    while (time.time() - start_time) < duration_seconds:
        data = task.read(number_of_samples_per_channel=samples_per_read)
        all_data.extend(data)

        display_buffer = np.roll(display_buffer, -samples_per_read)
        display_buffer[-samples_per_read:] = data

        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(time_axis, display_buffer)
        ax.set_xlabel("tiempo [s]")
        ax.set_ylabel("Tension [V]")
        ax.set_title(f"medición a {sample_rate} Hz")
        ax.set_ylim(display_buffer.min() - 0.1, display_buffer.max() + 0.1)
        ax.grid(True)
        plt.tight_layout()
        plt.show()

finally:
    task.stop()
    task.close()

all_data = np.array(all_data)
print(f"Collected {len(all_data)} samples ({len(all_data)/sample_rate:.2f} seconds)")

V = all_data
t = np.arange(len(V)) / sample_rate

In [ ]:
save_measurement(t, V, comment="Medición dinámica con adquisición continua")